# Create NSFC Awards (National Natural Science Foundation of China)

Creates NSFC awards from the official NSFC big-data portal completed-project (结题) database.

**Prerequisites:**
- Run `scripts/local/nsfc_to_s3.py` to harvest + upload the data first.
- ⚠️ **Geo-constraint:** `kd.nsfc.cn` is GFW/geo-restricted and is **unreachable from US/EU infra**. The harvest MUST run from a **China/Hong Kong IP**; the contractor produces the parquet in-region and hands it off. Admin only runs the upload + this notebook (not the harvest).

**Data source:** https://kd.nsfc.cn (POST /api/baseQuery/completionQueryResultsData, anonymous)
**S3 location:** `s3a://openalex-ingest/awards/nsfc/nsfc_projects.parquet`

**NSFC funder in OpenAlex:**
- funder_id: 4320321001
- display_name: "National Natural Science Foundation of China"
- ror_id: "https://ror.org/01h0zpd94"
- doi: "10.13039/501100001809"

**Input columns from nsfc_to_s3.py:**
- project_id -> internal NSFC id (dedup / landing page)
- funder_award_id -> ratify_no (批准号, real grant number cited in papers)
- display_name -> title (项目名称)
- funder_scheme -> project_type (面上项目 / 青年科学基金项目 / 重点项目 / ...)
- institution -> dependUnit (依托单位)
- given_name / family_name -> personInCharge (项目负责人), split
- amount (DOUBLE, CNY; source 万元 * 10000)
- start_year / start_date ; conclusion_year (-> end)
- subject_code / subject_dept (申请代码; dept A..H)

**Notes:** amount is real CNY (negative/zero -> NULL per §6.7, none observed). The source list API exposes no prose abstract, so `description` is NULL (keywords + subject_code are captured in the raw parquet but have no awards-schema field). start_date = approval year; end_date = conclusion year (both real).

**Priority 0 (precedence fix, 2026-06-05).** NSFC's own completed-project database is the authoritative source for NSFC grants, so it is inserted at priority 0 to outrank the Crossref acknowledgement-derived records (`crossref_work_funders` / `crossref_work.grants`, priority 2). The dedup is `PARTITION BY id ORDER BY priority ASC` where `id = hash(funder_id:funder_award_id)`; because NSFC `ratify_no` exactly matches the grant numbers cited in papers, ~126k NSFC grants collide with Crossref rows. At the original priority 209 the sparse Crossref shells won and hid the rich NSFC data; priority 0 makes the direct source win.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nsfc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nsfc/nsfc_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.nsfc_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.nsfc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nsfc_raw LIMIT 5;

## Step 2: Create NSFC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nsfc_awards
USING delta
AS
WITH
-- Get NSFC funder from OpenAlex by explicit funder_id
nsfc_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321001  -- National Natural Science Foundation of China
),

awards_transformed AS (
    SELECT
        -- Unique ID from funder_id:funder_award_id (ratify_no)
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        -- Display name = project title
        g.display_name as display_name,

        -- Description: source list API exposes no abstract -> NULL (keywords kept in raw only)
        CAST(NULL AS STRING) as description,

        -- Funder info
        f.funder_id,
        g.funder_award_id as funder_award_id,

        -- Amount (already CNY from nsfc_to_s3.py); 0/neg already NULLed upstream (6.7)
        TRY_CAST(g.amount AS DOUBLE) as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'CNY' ELSE NULL END as currency,

        -- Funder struct
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- Funding type from NSFC programme (funder_scheme)
        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优秀青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%创新研究群体%' THEN 'research'
            WHEN g.funder_scheme LIKE '%基础科学中心%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重大%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        -- Funder scheme = NSFC project type
        g.funder_scheme as funder_scheme,

        -- Provenance
        'nsfc_kd' as provenance,

        -- Dates: start = approval year, end = conclusion year (both real)
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(CONCAT(g.conclusion_year, '-12-31'), 'yyyy-MM-dd') as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.conclusion_year AS INT) as end_year,

        -- Lead investigator: NSFC gives PI given_name / family_name / affiliation
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation
        CASE
            WHEN (g.family_name IS NOT NULL AND TRIM(g.family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    g.given_name as given_name, g.family_name as family_name, CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        -- Co-lead / other investigators not exposed by the list API
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        -- Landing page = NSFC project detail page
        CONCAT('https://kd.nsfc.cn/finalDetails?id=', g.project_id) as landing_page_url,

        CAST(NULL AS STRING) as doi,

        -- Works API URL
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM openalex.awards.nsfc_raw g
    CROSS JOIN nsfc_funder f
    WHERE g.funder_award_id IS NOT NULL
      AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)

SELECT * FROM awards_transformed;


In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
-- NOTE: clear BOTH the old priority (209) and the new one (0) so a re-run after
-- the precedence fix doesn't leave stale priority-209 rows behind.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nsfc_kd' AND priority IN (0, 209);

-- Insert into openalex_awards_raw with priority.
-- Priority 209: NSFC's own completed-project database is authoritative for NSFC grants.
-- Restored to its original priority 209 on 2026-06-20 (oxjob #500) after the dedup
-- direction was flipped from ASC to DESC so HIGHER priority wins. Pre-flip workaround
-- (priority 0) is no longer needed — at priority 209, NSFC now properly outranks the
-- shells at priority 0 (crossref_work_funders, crossref_work.grants, datacite_work_funders,
-- etc.).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    209 as priority
FROM openalex.awards.nsfc_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_nsfc_awards FROM openalex.awards.nsfc_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year, end_year, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.nsfc_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.nsfc_awards GROUP BY funder.display_name ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.nsfc_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.nsfc_awards WHERE funder_scheme IS NOT NULL GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    ROUND(try_divide(COUNT(amount) * 100.0, COUNT(*)), 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.nsfc_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.nsfc_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.nsfc_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name ORDER BY grant_count DESC LIMIT 20;